# 02 · Lists

**How this notebook works**
- Run the setup cell below first. Then go top to bottom, running each cell with **Shift+Enter**.
- Gray **example** cells: run them, then change something and run again.
- **Try it** cells: your scratch space for each section.
- **Predict the output**: write your guess in the comment *before* running.
- **Exercises** at the bottom: write your code, run the cell, then run the `check(...)` cell under it.

In [ ]:
# ▶ Run this cell first (Shift+Enter). It loads the test helper and the same imports LeetCode gives you.
import sys, pathlib
sys.path.insert(0, str(next(p for p in (pathlib.Path.cwd() / "_primer", pathlib.Path.cwd() / "python-primer" / "_primer") if p.exists())))
from check import check, summary, lesson
lesson("ex02_lists")

import bisect, heapq, math, random
from collections import Counter, OrderedDict, defaultdict, deque
from functools import cache, lru_cache
from itertools import accumulate, combinations, groupby, pairwise, permutations, product
from typing import List, Optional

**Warm-up (5 min):** before reading, close everything and write down every trap you remember from lessons 01–01. Then check them against your *My traps* notes in those notebooks.

A Python list is Java's `ArrayList`: a dynamic array. It's also your stack.

## The operations you'll use daily

In [ ]:
nums = [3, 1, 2]
print(len(nums))           # 3
print(nums[0], nums[-1])   # 3 2        negative index counts from the end
nums.append(4)             # O(1)        push
print(nums.pop())          # O(1)        pop from the end, returns it
print(nums.pop(0))         # O(n)        shifts everything; use a deque instead
nums.insert(0, 9)          # O(n)        same problem
print(nums)
print(2 in nums)           # O(n)        linear scan; use a set for membership
print(nums.index(2))       # O(n)        raises ValueError if missing
print([0] * 4)             # n zeros
stack = [1, 2, 3]
print(stack[-1])           # peek

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Slicing

`nums[start:stop:step]`, stop exclusive, any part optional.

In [ ]:
nums = [10, 20, 30, 40, 50]
k = 2
print(nums[1:3])      # elements 1 and 2
print(nums[:k])       # first k
print(nums[-k:])      # last k
print(nums[::-1])     # reversed copy
copy = nums[:]        # shallow copy
copy.append(99)
print(nums, copy)     # the original didn't change

**Every slice is a new list and costs O(k).** `nums[1:]` inside a recursive call turns an O(n) algorithm into O(n²). Pass indices instead.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## "Modify nums in-place"

Many array problems check the list object you were given. Rebinding the name does nothing to the caller's list:

In [ ]:
def rotate_wrong(nums, k):
    nums = nums[-k:] + nums[:-k]     # WRONG: makes a new list, caller sees no change

def rotate_right(nums, k):
    nums[:] = nums[-k:] + nums[:-k]  # RIGHT: overwrites the contents of the same list

a = [1, 2, 3, 4, 5]
rotate_wrong(a, 2)
print("wrong:", a)
rotate_right(a, 2)
print("right:", a)
print([3, 1, 2].sort())              # sort() returns None

Also: `nums.sort()` sorts in place and **returns `None`**. `nums = nums.sort()` sets nums to `None`. `sorted(nums)` returns a new list.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Sorting with keys

In [ ]:
words = ["kiwi", "fig", "banana"]
words.sort(key=len)                       # by length
print(words)
pairs = [(1, "b"), (2, "a")]
pairs.sort(key=lambda p: p[1])            # by second element
print(pairs)
people = [("bo", 30), ("al", 25), ("cy", 30)]
people.sort(key=lambda p: (-p[1], p[0]))  # age descending, then name ascending
print(people)
intervals = [[5, 6], [1, 3], [1, 2]]
intervals.sort()                          # lists/tuples sort element by element
print(intervals)
print(max([3, -7, 5], key=abs))           # key works on max/min too

Python's sort is stable and O(n log n). Negating a number in the key is the standard way to get descending order on one field.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Iteration helpers

In [ ]:
nums = ["a", "b", "c"]
for i, x in enumerate(nums):          # index and value, instead of range(len(nums))
    print(i, x)
for a, b in zip([1, 2, 3], ["x", "y"]):   # walk two lists together, stops at the shorter
    print(a, b)
for x in reversed(nums):              # no copy
    print(x)
matrix = [[1, 2, 3], [4, 5, 6]]
print(list(zip(*matrix)))             # transpose: rows become column tuples

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Comprehensions

In [ ]:
nums = [1, 2, 3, 4]
n = len(nums)
squares = [x * x for x in nums]
evens = [x for x in nums if x % 2 == 0]
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(squares)
print(evens)
print(pairs)

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## 2D grids, the classic trap

In [ ]:
rows, cols = 3, 2
grid = [[0] * cols for _ in range(rows)]   # RIGHT: each row is its own list
bad = [[0] * cols] * rows                  # WRONG: every row is the same list object
grid[0][0] = 1
bad[0][0] = 1
print("right:", grid)
print("wrong:", bad)                       # column 0 of every row became 1

Neighbors in a grid:

In [ ]:
grid = [[1, 2, 3],
        [4, 5, 6]]
rows, cols = len(grid), len(grid[0])
r, c = 0, 1
for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
    r2, c2 = r + dr, c + dc
    if 0 <= r2 < rows and 0 <= c2 < cols:
        print("neighbor", (r2, c2), "=", grid[r2][c2])

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Copies in backtracking

In [ ]:
res, path = [], [1]
res.append(path)       # WRONG: stores a reference
res.append(path[:])    # RIGHT: stores a snapshot
path.append(2)
print(res)             # the first entry changed along with path; the snapshot didn't

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Predict the output

For each one: type your guess into the comment line, **then** run the cell. The output is the answer. If you were wrong, figure out why before moving on.

**1.** What does this print?

In [ ]:
# My guess: 
a = [1, 2, 3]
b = a
b.append(4)
print(a)

**2.** What does this print?

In [ ]:
# My guess: 
print([1, 2, 3, 4, 5][1:-1])

**3.** What does this print?

In [ ]:
# My guess: 
g = [[0] * 2] * 2
g[1][0] = 5
print(g)

**4.** What does this print?

In [ ]:
# My guess: 
print(sorted([(2, 'b'), (1, 'z'), (2, 'a')]))

**5.** What does this print?

In [ ]:
# My guess: 
nums = [3, 1, 2]
print(nums.sort())

<details><summary>Why (open after running all of them)</summary>

1. `[1, 2, 3, 4]` (`b = a` copies the reference, not the list)
2. `[2, 3, 4]`
3. `[[5, 0], [5, 0]]` (both rows are the same list)
4. `[(1, 'z'), (2, 'a'), (2, 'b')]`
5. `None`
</details>

---
# Exercises

Write your code in each cell (replace `raise NotImplementedError`), run it, then run the `check(...)` cell below it. `ok` means done; `FAIL` shows what was expected; `TODO` means not started.

### 1. `rotate_in_place` · LeetCode 189

Rotate nums right by k steps IN PLACE. Return nothing. k can be larger than len(nums). One line with slice assignment is fine.

```
nums = [1, 2, 3, 4, 5, 6, 7]
rotate_in_place(nums, 3)
nums is now [5, 6, 7, 1, 2, 3, 4], and the function returns None
```

In [ ]:
def rotate_in_place(nums: list, k: int) -> None:
    raise NotImplementedError

In [ ]:
check("rotate_in_place")

### 2. `make_grid`

rows x cols grid where every row is an independent list.

```
make_grid(3, 2)  ->  [[0, 0], [0, 0], [0, 0]]
changing g[0][0] must not change g[1][0]
```

In [ ]:
def make_grid(rows: int, cols: int, fill=0) -> list:
    raise NotImplementedError

In [ ]:
check("make_grid")

### 3. `sort_people`

people is a list of (name, age). Return a new list sorted by age descending, then name ascending. Use one sorted() call with a key.

```
sort_people([("bo", 30), ("al", 25), ("cy", 30)])
  ->  [("bo", 30), ("cy", 30), ("al", 25)]
```

In [ ]:
def sort_people(people: list) -> list:
    raise NotImplementedError

In [ ]:
check("sort_people")

### 4. `prefix_sums`

Return p with len(nums) + 1 entries where p[i] = sum(nums[:i]). Then sum(nums[i:j]) == p[j] - p[i]. Build it in O(n).

```
prefix_sums([])  ->  [0]
```

In [ ]:
def prefix_sums(nums: list) -> list:
    raise NotImplementedError

In [ ]:
check("prefix_sums")

### 5. `positive_positions`

List of (index, value) for every value > 0, using enumerate in a comprehension.

```
positive_positions([0, 5, -2, 7])  ->  [(1, 5), (3, 7)]
```

In [ ]:
def positive_positions(nums: list) -> list:
    raise NotImplementedError

In [ ]:
check("positive_positions")

### 6. `transpose`

Transpose a rectangular matrix, returning a list of lists. Try zip(*matrix).

```
transpose([[1, 2, 3], [4, 5, 6]])  ->  [[1, 4], [2, 5], [3, 6]]
```

In [ ]:
def transpose(matrix: list) -> list:
    raise NotImplementedError

In [ ]:
check("transpose")

### 7. `all_subsets_wrong_vs_right`

Return every subset of nums (order of subsets: as produced below). Fill in the one line marked FIX so the result is correct.

```
all_subsets_wrong_vs_right([1, 2])  ->  [[1, 2], [1], [2], []]
```

In [ ]:
def all_subsets_wrong_vs_right(nums: list) -> list:
    res, path = [], []

    def backtrack(i):
        if i == len(nums):
            raise NotImplementedError  # FIX: record the current path correctly
            return
        path.append(nums[i])
        backtrack(i + 1)
        path.pop()
        backtrack(i + 1)

    backtrack(0)
    return res

In [ ]:
check("all_subsets_wrong_vs_right")

---
## Wrap-up

In [ ]:
summary()

## My traps

*Double-click to edit.* Write the 3–5 things from this lesson that surprised you or that you got wrong, in your own words. You'll use these in the next warm-up and the exit test.

1. 
2. 
3. 

**Done?** All `ok` in the summary → commit and sync → tell Claude **"lesson 02 done."**